<!-- circuitkit-logo -->
<p align="center">
  <picture>
    <source media="(prefers-color-scheme: dark)" srcset="../../docs/assets/circuitkit-logo-white.png">
    <img src="../../docs/assets/circuitkit-logo-black.png" width="220" alt="CircuitKIT">
  </picture>
</p>

# 💻 CLI and YAML Pipelines

CircuitKIT provides a full CLI for terminal and CI/CD workflows. Every Python API operation has a CLI equivalent.

This notebook demonstrates:
- **Section A:** Discovery via `circuitkit discover`
- **Section B:** Inspecting artifacts with `circuitkit inspect`
- **Section C:** Pruning and exporting via CLI
- **Section D:** YAML task configs with `circuitkit discover-yaml`
- **Section E:** Full YAML pipeline with `circuitkit run`
- **Section F:** Data utilities

- **Model:** GPT-2 (CPU-friendly)
- **Runtime:** ~5 min

---

## Setup

In [1]:
# For Colab: uncomment and run the setup cells from 00_colab_setup.ipynb
# For local: pip install circuitkit

import sys, os
print("python:", sys.executable)
print("expected CLI dir:", os.path.dirname(sys.executable))   # e.g. ~/circuitkit/.venv/bin
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
!which circuitkit    # should now resolve

python: <repo>/.venv/bin/python3.12
expected CLI dir: <repo>/.venv/bin
<repo>/.venv/bin/circuitkit


In [2]:
# Verify CLI is installed
!circuitkit --help

Usage: circuitkit [OPTIONS] COMMAND [ARGS]...

  CircuitKIT: Circuit Discovery and Analysis for Transformer Models

Options:
  -v, --verbose      Enable verbose output
  -c, --config TEXT  Path to configuration file
  --help             Show this message and exit.

Commands:
  benchmark        Run comprehensive benchmarks comparing circuit methods...
  data             Custom-data tools: worthiness check, prepare...
  debug            Debugging and profiling tools for CircuitKIT.
  discover         Run circuit discovery on a model using built-in task...
  discover-smart   Run memory-efficient circuit discovery with automatic...
  discover-yaml    Run circuit discovery using a YAML task configuration
  evaluate         Evaluate circuit faithfulness for a discovered/pruned...
  export           Apply an intervention to a model and export a...
  heal             Fine-tune a pruned model using circuit-guided LoRA...
  inspect          Inspect a circuit artifact (.pt) and print a summary.
 

---
## Section A: Discovery via CLI

The `circuitkit discover` command mirrors `Pipeline.discover()`.

In [3]:
!circuitkit discover \
    --model gpt2 \
    --algorithm eap-ig \
    --task ioi \
    --level node \
    --scope both \
    --sparsity 0.3 \
    --batch-size 4 \
    --num-examples 32 \
    --ig-steps 5 \
    --output ./results/cli_demo/circuit.pt

Starting circuit discovery...
Model: gpt2
Algorithm: EAP-IG
Task: ioi
Level: node
Sparsity: 0.3
Output: ./results/cli_demo/circuit.pt
06:52:49 • ==================================================━━━━━━━   0%
06:52:49 • Starting: Circuit Discovery━━━━━━━━━━━━━━━━━━━━━━━━━━   0%
06:52:49 • ==================================================━━━   0%
06:52:49 • [1/4] Loading and validating configuration━━━━━━━━━━━   0%
⠹ Running discovery... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0%06:52:49 ⚙ Configuration validated successfully.
06:52:49 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
06:52:49 • [2/4] Setting up model (model=gpt2)━━━━━━━━━━━━━━━━━━   0%
⠇ Running discovery... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0%`torch_dtype` is deprecated! Use `dtype` instead!
⠙ Running discovery... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0%2026-07-23 06:52:51 - root - WARNING - With reduced precision, it is advised to use `from_pretrained_no_processing` instead of `from_pretrained`.

---
## Section B: Inspect the Artifact

`circuitkit inspect` loads a `.pt` artifact and prints a structured summary.

In [4]:
!circuitkit inspect ./results/cli_demo/circuit.pt

        Circuit:         
./results/cli_demo/circui
          t.pt           
┏━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Property     ┃ Value  ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━┩
│ Level        │ node   │
│ Nodes        │ 46     │
│ Algorithm    │ eap-ig │
│ Task         │ ioi    │
│ Model        │ gpt2   │
│ Scored Nodes │ 156    │
└──────────────┴────────┘
 Top 10 Nodes by  
      Score       
┏━━━━━━━┳━━━━━━━━┓
┃ Node  ┃  Score ┃
┡━━━━━━━╇━━━━━━━━┩
│ MLP 0 │ 1.3359 │
│ A8.10 │ 0.0762 │
│ A3.0  │ 0.0579 │
│ A0.10 │ 0.0559 │
│ A6.9  │ 0.0547 │
│ A5.5  │ 0.0486 │
│ A7.3  │ 0.0430 │
│ A10.7 │ 0.0417 │
│ A11.3 │ 0.0398 │
│ A0.1  │ 0.0388 │
└───────┴────────┘


---
## Section C: Prune and Export via CLI

The CLI `prune` command loads the model, applies the circuit-guided pruning mask, and writes a HuggingFace checkpoint in one shot.

In [5]:
!circuitkit prune \
    --model gpt2 \
    --artifact ./results/cli_demo/circuit.pt \
    --sparsity 0.3 \
    --scope both \
    --precision float32 \
    --output ./results/cli_demo/pruned_checkpoint

Loading model: gpt2
`torch_dtype` is deprecated! Use `dtype` instead!
Loaded pretrained model gpt2 into HookedTransformer
Pruning gpt2 at sparsity=0.3 scope=both
06:53:06 • Starting structural masking: eap-ig scores on gpt2 (inplace=False)
06:53:06 • Masking 46 nodes: ['A8.9', 'A7.0', 'A5.1', 'A3.10', 'A4.9'] + 41 more
06:53:06 ✓ Masking complete. Effective sparsity: 5.55%
✓ Checkpoint exported to: ./results/cli_demo/pruned_checkpoint


In [6]:
# List the exported checkpoint files
!ls -la ./results/cli_demo/pruned_checkpoint/

total 641604
drwxr-xr-x 2 root root      4096 Jul 23 06:53 .
drwxr-xr-x 3 root root      4096 Jul 23 06:53 ..
-rw-r--r-- 1 root root       874 Jul 23 06:53 config.json
-rw-r--r-- 1 root root    456318 Jul 23 06:53 merges.txt
-rw------- 1 root root 652163888 Jul 23 06:53 model.safetensors
-rw-r--r-- 1 root root        99 Jul 23 06:53 special_tokens_map.json
-rw-r--r-- 1 root root       475 Jul 23 06:53 tokenizer_config.json
-rw-r--r-- 1 root root   3557680 Jul 23 06:53 tokenizer.json
-rw-r--r-- 1 root root    798156 Jul 23 06:53 vocab.json


---
## Section D: YAML Task Discovery

For custom data, you can define a task in a YAML file and discover circuits without writing Python code. `YAMLTaskLoader` expects three blocks:

- `source` — `type` (`csv`/`jsonl`/`hf`) and `path` (or `dataset_id` for `hf`)
- `schema` — maps `prompt` and `answer` (required) to column names; `choices` is optional
- `corruption` — optional. `strategy` is one of `entity_swap`, `token_swap`, `paraphrase`, `distractor`, `role_swap`, with an optional `config` block
- `metric` — optional, defaults to `logit_diff` (required for EAP-family algorithms; `accuracy` is non-differentiable and reporting-only)

First, create a sample CSV and YAML config inline. We use `token_swap` corruption to automatically generate corrupted prompts from the clean ones.

In [7]:
%%writefile ./results/cli_demo/sample_task.csv
question,answer
"The capital of France is"," Paris"
"The capital of Japan is"," Tokyo"
"The capital of Italy is"," Rome"
"The capital of Germany is"," Berlin"
"The capital of Spain is"," Madrid"
"The capital of Egypt is"," Cairo"
"The capital of Canada is"," Ottawa"
"The capital of Brazil is"," Brasilia"

Overwriting ./results/cli_demo/sample_task.csv


In [8]:
%%writefile ./results/cli_demo/sample_task.yaml
name: capital_cities
source:
  type: csv
  path: ./results/cli_demo/sample_task.csv
schema:
  prompt: question
  answer: answer
corruption:
  strategy: token_swap
metric: logit_diff

Overwriting ./results/cli_demo/sample_task.yaml


In [9]:
!circuitkit discover-yaml \
    --model gpt2 \
    --task-yaml ./results/cli_demo/sample_task.yaml \
    --algorithm eap-ig \
    --level node \
    --sparsity 0.2 \
    --num-examples 8 \
    --output ./results/cli_demo/yaml_circuit.pt

06:53:18 • Loaded task 'capital_cities' from YAML: results/cli_demo/sample_task.yaml
✓ Loaded task from YAML: capital_cities
Starting YAML-based circuit discovery...
Model: gpt2
Task: capital_cities
YAML: ./results/cli_demo/sample_task.yaml
Algorithm: EAP-IG
Output: ./results/cli_demo/yaml_circuit.pt
06:53:18 • ==================================================━━━━━━━   0%
06:53:18 • Starting: Circuit Discovery━━━━━━━━━━━━━━━━━━━━━━━━━━   0%
06:53:18 • ==================================================━━━   0%
06:53:18 • [1/4] Loading and validating configuration━━━━━━━━━━━   0%
⠙ Running discovery... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0%06:53:18 ⚙ Configuration validated successfully.
06:53:18 ⚙ Config: EAP-IG on capital_cities task, node level, 20% sparsity
06:53:18 • [2/4] Setting up model (model=gpt2)━━━━━━━━━━━━━━━━━━   0%
⠏ Running discovery... ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━   0%`torch_dtype` is deprecated! Use `dtype` instead!
⠧ Running discovery... ━━━━━━━━━━━

---
## Section E: Full YAML Pipeline

The `circuitkit run` command executes a complete pipeline from a single YAML config. It supports: model, discovery, evaluate, applications, export, benchmark, and visualize.

Key YAML semantics:
- `evaluate.enabled` defaults to `True` when the `evaluate:` key is present
- `applications` is a list of intervention steps
- Steps after discovery are non-fatal — failures print warnings and continue

In [10]:
%%writefile ./results/cli_demo/pipeline.yaml
model: gpt2
task: ioi
precision: float32
output_dir: ./results/cli_demo/pipeline_out

discovery:
  algorithm: eap-ig
  level: node
  sparsity: 0.3
  n_examples: 32
  batch_size: 4

evaluate:
  pillars:
    - patching
    - ablation
  n_examples: 32

applications:
  - type: prune
    sparsity: 0.3
    scope: both

export:
  path: ./results/cli_demo/pipeline_out/checkpoint
  intervention: pruning

visualize:
  mode: graph
  output: ./results/cli_demo/pipeline_out/circuit.html

Overwriting ./results/cli_demo/pipeline.yaml


In [11]:
!circuitkit run ./results/cli_demo/pipeline.yaml

Running pipeline: ./results/cli_demo/pipeline.yaml
  Model: gpt2 | Task: ioi
Step: discovery
`torch_dtype` is deprecated! Use `dtype` instead!
Loaded pretrained model gpt2 into HookedTransformer
06:53:31 • ==================================================
06:53:31 • Starting: Circuit Discovery
06:53:31 • ==================================================
06:53:31 • [1/4] Loading and validating configuration
06:53:31 ⚙ Configuration validated successfully.
06:53:31 ⚙ Config: EAP-IG on ioi task, node level, 30% sparsity
06:53:31 • [2/4] Setting up model (model=gpt2)
Loaded pretrained model gpt2 into HookedTransformer
06:53:34 🔧 Model loading: 2.84s
06:53:34 🔧 Model: gpt2 (163.0M params, cuda device)
06:53:34 • [3/4] Running discovery algorithm (algorithm=eap-ig)
06:53:34 • Starting EAP-IG discovery algorithm
06:53:34 • Loading cached IOI data for gpt2 from cache/ioi/ioi_gpt2_32_seed42.csv
100%|████████████████████████████████████████████| 8/8 [00:02<00:00,  3.21it/s]
06:53:37 • Converte

---
## Section F: Data Utilities

The `circuitkit data` subcommands help prepare and validate custom datasets.

In [12]:
# List available data shapes
!circuitkit data shapes

  mcq               MCQAdapter                default_strategy=mcq_choice_swap
      General MCQ datasets: builds a single clean prompt listing all choices and the correct-let
  pairwise          PairwiseAdapter           default_strategy=None
      CrowS-Pairs / StereoSet / similar datasets that supply sent_more and sent_less columns dir
  conversational    ConversationalAdapter     default_strategy=resample
      Multi-turn chat datasets (ShareGPT / OpenAI messages / OpenChat / PIPPA): each (history+us
  instruction       InstructionAdapter        default_strategy=resample
      Single-turn instruction-tuning datasets (Alpaca / Dolly / OASST): {instruction, input?, ou
  forget_retain     ForgetRetainAdapter       default_strategy=resample
      Unlearning benchmarks with explicit forget/retain splits (TOFU, MUSE, WMDP). Caller suppli
  code              CodeAdapter               default_strategy=code_syntax_corrupt
      Code-generation datasets. Extracts the function signature + doc

In [13]:
# List available data strategies
!circuitkit data strategies

  benign_rewrite          contract=unknown
      Symbolic harmful->benign keyword rewrite for safety datasets. No external LLM required. Fo
  code_syntax_corrupt     contract=preserve
      Corrupt a code prompt by flipping a comparison operator, renaming a parameter, or swapping
  entity_swap             contract=preserve
      Swap a salient capitalised entity in the clean prompt for a same-length alternative. Lengt
  final_answer_swap       contract=preserve
      Math-reasoning corruption: perturbs the operand of the final calculator step in a chain-of
  instruction_swap        contract=preserve
      Swap the directive verb in an instruction-style prompt (generate <-> list, explain <-> sum
  llm_counterfactual      contract=unknown
      Use a small instruction-tuned LLM to rewrite the clean prompt into a counterfactual. Pass 
  logical_negation        contract=unknown
      Flip the logical polarity of the prompt: insert 'not' after an auxiliary verb, or remove a
  math_step_corr

---
## CLI Command Reference

| Command | Description |
|---------|-------------|
| `circuitkit discover` | Run circuit discovery on a built-in task |
| `circuitkit discover-yaml` | Discover using a YAML task config |
| `circuitkit evaluate` | Evaluate a circuit artifact |
| `circuitkit inspect` | Print artifact summary |
| `circuitkit prune` | Prune + export in one shot |
| `circuitkit quantize` | Quantize + export in one shot |
| `circuitkit export` | Export an intervened model |
| `circuitkit run` | Full pipeline from YAML |
| `circuitkit benchmark` | Run lm-eval harness |
| `circuitkit list-models` | List supported models |
| `circuitkit discover-smart` | Memory-aware discovery |
| `circuitkit data *` | Data preparation utilities |